# Setup


In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
from logging import INFO

from torch.backends import cudnn

# enforce more deterministic behavior
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
cudnn.deterministic = True
cudnn.benchmark = False

import pandas as pd

sys.path.append("../..")

from pneuma_seeker.core.materializer.main import Materializer
from pneuma_seeker.model.interface.model_factory import get_llm, get_embed_model
from pneuma_seeker.provenance.graph import ProvenanceGraph
from pneuma_seeker.utils.logger import setup_logger
from pneuma_seeker.utils.config import Config

In [ ]:
llm_path = "o4-mini"
embed_model_path = "../model/weight/bge-base"
llm = get_llm(llm_path, Config("../../../.env"))(llm_path)
embed_model = get_embed_model()(embed_model_path)

logger = setup_logger(
    name="processor_logger",
    log_path=os.path.join(".", "log"),
    level=INFO,
    max_bytes=10_000_000,
    backup_count=5,
)
materializer = Materializer(llm, logger, embed_model, ["tag"], ProvenanceGraph(logger))

# Experiments


In [ ]:
T = {
    "satscores": pd.DataFrame(columns=["cname", "numtsttakr", "population_over_2m"])
}
column_descriptions = {
    "satscores": {
        "cname": "Name of the county where the school is located",
        "numtsttakr": "Number of SAT test takers at the school",
        "population_over_2m": "Whether the county has population over 2 million (derived)",
    }
}
Q = [
    "SELECT SUM(numtsttakr) AS total_test_takers\nFROM satscores\nWHERE population_over_2m = true"
]

In [ ]:
materializer.materialize_T(
    T,
    column_descriptions,
    Q,
)

In [ ]:
materializer.state.intermediate_tables